# debug_armt_state — ARMT associative state-size testbed

Sibling of `debug_rmmv5.ipynb`, but for **ARMT** (`run_original_armt_on_kv_retrieval-v3-gen.py`).

**Do not trust prose claims** about ARMT's recurrent capacity. Rebuild the exact model the
run script builds, then read the *actual* `W_mem` geometry off the live `AssociativeLayerWrapper`s.

Why this matters for the RMM-vs-ARMT comparison: ARMT's per-layer associative state is **not**
`d_mem × d_model`. The DPFP feature map (ν=3) expands the key dim to `d_key = 2·ν·d_mem = 6·d_mem`,
so `W_mem` per layer is `d_key × d_model / n_heads`. With the baseline `d_mem=32, d_model=128, n_heads=1`
that is **192×128 = 24,576 floats/layer** — far larger than the RMM v6p4 `id-ca` GDN state (~512/layer).
Match this before attributing any gap to "GDN vs fast-weights".

In [ ]:
import os, sys, inspect
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("cwd:", os.getcwd())
import torch
from transformers import AutoConfig, AutoTokenizer
TOKENIZER_PATH = "./tokenizers/kv_alphabet_62/"   # as in run_original_armt_on_kv_retrieval-v3-gen.py

In [ ]:
def build_armt_model(d_mem=32, n_mem_tokens=8, n_heads=1, n_embd=128, n_layer=4, n_head=4,
                     use_denom=True, correction=True, gating=False):
    """Replicates run_original_armt_on_kv_retrieval-v3-gen.py model creation (llama base).
    n_heads is the ASSOCIATIVE head count (ARMTConfig.n_heads); n_head is the base
    transformer's attention heads. The run script leaves n_heads/use_denom/gating at
    ARMTConfig defaults (1 / True / False) and sets d_mem + num_mem_tokens."""
    tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    config = AutoConfig.from_pretrained("NousResearch/Llama-3.2-1B")
    config.num_hidden_layers   = n_layer
    config.num_attention_heads = n_head
    config.num_key_value_heads = n_head
    config.hidden_size         = n_embd
    config.head_dim            = n_embd // n_head
    config.intermediate_size   = n_embd * 4
    config.torch_dtype = "float32"
    config.vocab_size  = tok.vocab_size
    config.pad_token_id = tok.convert_tokens_to_ids("[PAD]")
    config.bos_token_id = tok.convert_tokens_to_ids("[BOS]")
    config.eos_token_id = tok.convert_tokens_to_ids("[EOS]")

    from modeling_armt.huggingface import ARMTForCausalLMv2, ARMTConfig
    rmt_config = ARMTConfig()
    rmt_config.base_model_config = config
    rmt_config.num_mem_tokens = n_mem_tokens
    rmt_config.d_mem = d_mem
    rmt_config.n_heads = n_heads
    rmt_config.use_denom = use_denom
    rmt_config.correction = correction
    rmt_config.gating = gating
    rmt_config.max_n_segments = 10
    rmt_config.think_token_id  = tok.convert_tokens_to_ids('[THINK]')
    rmt_config.answer_token_id = tok.convert_tokens_to_ids('[ANSWER]')
    rmt_config.bos_token_id    = tok.convert_tokens_to_ids('[BOS]')
    rmt_config.eos_token_id    = tok.convert_tokens_to_ids('[EOS]')
    model = ARMTForCausalLMv2(rmt_config)
    return model, rmt_config, dict(d_mem=d_mem, n_mem_tokens=n_mem_tokens, n_heads=n_heads,
                                   n_embd=n_embd, n_layer=n_layer)

In [ ]:
def _armt_layers(model):
    # AssociativeMemoryCell.get_layers() returns the live AssociativeLayerWrapper list.
    return list(model.armt.memory_cell.get_layers())

def inspect_armt(model):
    """Read ACTUAL associative geometry/state off the live modules."""
    layers = _armt_layers(model)
    L0 = layers[0]
    Wm = L0.W_mem
    out = dict(
        cls=L0.__class__.__name__,
        n_heads=L0.n_heads, d_mem=L0.d_mem, d_model=L0.d_model,
        d_key=L0.d_key, nu=getattr(L0.phi, 'nu', None), use_denom=L0.use_denom,
        correction=L0.correction, num_mem_tokens=L0.num_mem_tokens,
        W_mem_shape=tuple(Wm.shape), W_mem_numel=int(Wm.numel()),
    )
    z_numel = int(L0.z.numel()) if getattr(L0, 'use_denom', False) else 0
    if z_numel:
        out['z_shape'] = tuple(L0.z.shape)
    out['z_numel'] = z_numel
    out['recurrent_state_numel_per_layer'] = int(Wm.numel()) + z_numel
    out['recurrent_state_numel_total']     = out['recurrent_state_numel_per_layer'] * len(layers)
    mc = model.armt.memory_cell
    out['mem_bank_shape'] = tuple(mc.memory.shape)
    out['assoc_params_per_layer'] = int(sum(
        p.numel() for n, p in L0.named_parameters()
        if any(k in n for k in ['W_mq', 'W_mk', 'W_mv', 'W_mb'])))
    return out

## ARMT state formula (verify against the live read below)

```
DPFP feature map:   d_key = 2 · ν · d_mem            # ν = 3  →  d_key = 6 · d_mem
W_mem per layer  =  n_heads · (d_key // n_heads) · (d_model // n_heads)
                 =  d_key · d_model / n_heads        # when n_heads | d_key, d_model
z (denom)        =  d_key                            # if use_denom
state per layer  =  W_mem + z
state total      =  state_per_layer · n_layer
```
The mem-token bank (`memory`, shape `num_mem_tokens × d_model`) is re-initialized fresh each
segment — it is NOT the cross-segment carry. Only `W_mem` / `z` persist across segments.

In [ ]:
# Baseline config from run_original_armt_on_kv_retrieval-7tps-baseline.sh: d_mem=32, n_mem_tokens in {1,2,4,8}
model, cfg, req = build_armt_model(d_mem=32, n_mem_tokens=8)
info = inspect_armt(model); del model
for k, v in info.items():
    print(f"  {k:34s}: {v}")

In [ ]:
# Sweep d_mem (the associative key width) and n_mem_tokens. Note: state is INDEPENDENT of
# n_mem_tokens — mem tokens are the per-segment write probes, the carried state is W_mem.
print(f"{'d_mem':>6} {'n_mem':>6} {'d_key':>6} {'W_mem/layer':>12} {'+z/layer':>10} {'state/layer':>12} {'state×L':>10}")
for d_mem in [16, 32, 64, 128]:
    for n_mem in [4, 8]:
        m, _, _ = build_armt_model(d_mem=d_mem, n_mem_tokens=n_mem)
        g = inspect_armt(m); del m
        print(f"{d_mem:>6} {n_mem:>6} {g['d_key']:>6} {g['W_mem_numel']:>12} "
              f"{g['z_numel']:>10} {g['recurrent_state_numel_per_layer']:>12} {g['recurrent_state_numel_total']:>10}")

## Capacity match vs RMM v6p4 GDN (the comparison you actually want fair)

RMM v6p4 `id-ca` GDN state per layer = `num_heads · head_k_dim · head_v_dim`, with
`head_k_dim = state_size // n_head` and `head_v_dim = head_k_dim · expand_v`.
Build the RMM model and read it off the live module (needs the `fla` env). Then size the GDN
(or shrink ARMT `d_mem`) so the two per-layer states match.

In [ ]:
def build_rmm_v6p4(state_size=32, n_head=4, n_embd=128, n_layer=4, expand_v=2.0,
                   num_memory_vectors=32, write_mode='cross_attn', read_mode='identity',
                   num_compress_heads=4, conv_kernel=4, module='v6p4'):
    """Replicates run_rmm_on_kv_retrieval-v6p4.py (id-ca) model creation."""
    tok = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    config = AutoConfig.from_pretrained("NousResearch/Llama-3.2-1B")
    config.num_hidden_layers   = n_layer
    config.num_attention_heads = n_head
    config.num_key_value_heads = n_head
    config.hidden_size         = n_embd
    config.head_dim            = n_embd // n_head
    config.intermediate_size   = n_embd * 4
    config.torch_dtype = "float32"
    config.vocab_size  = tok.vocab_size
    config.pad_token_id = tok.convert_tokens_to_ids("[PAD]")
    config.bos_token_id = tok.convert_tokens_to_ids("[BOS]")
    config.eos_token_id = tok.convert_tokens_to_ids("[EOS]")
    mod = __import__(f"modeling_rmt.huggingface_rmm_{module}", fromlist=["x"])
    head_dim = state_size // n_head
    rmm_config = mod.RecurrentMemoryConfig(
        base_model_config=config, fla_layer_name="GatedDeltaNet",
        num_heads=n_head, head_dim=head_dim, expand_v=expand_v,
        conv_size=conv_kernel, use_short_conv=True,
        num_memory_vectors=num_memory_vectors, write_mode=write_mode, read_mode=read_mode,
        write_value_dim=None, num_memory_heads=1, num_compress_heads=num_compress_heads,
        use_parallel_prefill=True, thread_memory=True, max_n_segments=10,
        think_token_id=tok.convert_tokens_to_ids("[THINK]"),
        answer_token_id=tok.convert_tokens_to_ids("[ANSWER]"),
        bos_token_id=tok.convert_tokens_to_ids("[BOS]"),
        eos_token_id=tok.convert_tokens_to_ids("[EOS]"))
    return mod.RecurrentMemoryBase(rmm_config)

def rmm_gdn_state(model):
    cell = next(m for m in model.modules() if m.__class__.__name__ == 'RecurrentMemoryCell')
    g = (cell.model.model.layers if hasattr(cell.model, 'model') else cell.model.transformer.h)[0].fla_layer
    nh, hk, hv = g.num_heads, g.head_k_dim, g.head_v_dim
    return dict(num_heads=nh, head_k_dim=hk, head_v_dim=hv, state_numel_per_layer=nh*hk*hv)

In [ ]:
# ARMT baseline state
m, _, _ = build_armt_model(d_mem=32, n_mem_tokens=8); armt = inspect_armt(m); del m
armt_state = armt['recurrent_state_numel_per_layer']
print(f"ARMT  (d_mem=32, n_heads=1): state/layer = {armt_state}")

try:
    rm = build_rmm_v6p4(state_size=32, n_head=4, expand_v=2.0); g = rmm_gdn_state(rm); del rm
    print(f"RMM v6p4 id-ca (ss=32,H4,exp2): GDN state/layer = {g['state_numel_per_layer']}  {g}")
    print(f"ratio ARMT / RMM = {armt_state / g['state_numel_per_layer']:.1f}x")
    print()
    print("To match: raise RMM state (state_size / expand_v / n_head) OR lower ARMT d_mem.")
    print(f"  e.g. ARMT d_mem so that 6*d_mem*128 + 6*d_mem ~= {g['state_numel_per_layer']}")
except Exception as e:
    print("RMM build skipped (needs the fla env):", type(e).__name__, e)